In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
%pip install pyspark

In [23]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

# Create and config Spark
config = SparkConf().setAppName("bai02").setMaster("local[*]")
sparkContext = SparkContext.getOrCreate(conf=config)
sparkSession = SparkSession.builder.appName("bai02").getOrCreate()

In [24]:
data_path = "/content/drive/MyDrive/data/"

# Read file
movies_data = sparkContext.textFile(data_path + "movies.txt")

ratings1_data = sparkContext.textFile(data_path + "ratings_1.txt")
ratings2_data = sparkContext.textFile(data_path + "ratings_2.txt")

In [25]:
def splition_movie_with_genres(line):
    parts = line.split(',', 2)
    movie_id = int(parts[0])
    title = parts[1]
    genres = parts[2] if len(parts) > 2 else ""
    # Split genre
    genre_list = [genre.strip() for genre in genres.split('|') if genre.strip()]
    return (movie_id, title, genre_list)

movies_map = movies_data.map(splition_movie_with_genres)

# Create dictionary to retrieve
movies_dict = movies_map.map(lambda x: (x[0], (x[1], x[2]))).collectAsMap()

# Create dataset that contains (movie_id, genre) for each movie
movie_genres = movies_map.flatMap(lambda x: [(x[0], genre) for genre in x[2]])


In [28]:
# Split ratings: UserID, MovieID, Rating, Timestamp
def splition_rating(line):
    parts = line.split(',')
    user_id = int(parts[0])
    movie_id = int(parts[1])
    rating = float(parts[2])
    timestamp = int(parts[3])
    return (movie_id, rating)

ratings1_map = ratings1_data.map(splition_rating)
ratings2_map = ratings2_data.map(splition_rating)

# Combine 2 file
total_ratings = ratings1_map.union(ratings2_map)

In [29]:
# Join to create (movie_id, (rating, genre))
ratings_and_genres = total_ratings.join(movie_genres)

# change to (genre, rating)
genre_and_rating = ratings_and_genres.map(lambda x: (x[1][1], x[1][0]))

genre_ratings_add_count = genre_and_rating.map(lambda x: (x[0], (x[1], 1)))

# Reduce key
genre_analysis = genre_ratings_add_count.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))


In [30]:
# Calculate average rating for each genre
def calculation_average(record):
    genre, (sum_ratings, count) = record
    average_rating = sum_ratings / count
    return (genre, (average_rating, count))

genre_results = genre_analysis.map(calculation_average)

total_genres = genre_results.collect()
for genre, (avg_point, count) in total_genres:
    print(f"{genre} - AverageRating: {avg_point:.2f} (TotalRatings: {count})")

Sci-Fi - AverageRating: 3.73 (TotalRatings: 54)
Action - AverageRating: 3.71 (TotalRatings: 54)
Drama - AverageRating: 3.76 (TotalRatings: 128)
Family - AverageRating: 3.67 (TotalRatings: 18)
Biography - AverageRating: 3.56 (TotalRatings: 25)
Horror - AverageRating: 4.00 (TotalRatings: 2)
Fantasy - AverageRating: 3.86 (TotalRatings: 29)
Thriller - AverageRating: 3.70 (TotalRatings: 27)
Mystery - AverageRating: 4.00 (TotalRatings: 2)
Adventure - AverageRating: 3.63 (TotalRatings: 83)
Film-Noir - AverageRating: 4.36 (TotalRatings: 7)
Crime - AverageRating: 3.81 (TotalRatings: 42)
